# Get Photometry of all white dwarfs in the Gentile Fusillo Catalogue and then get PanSTARRS and probably a lot of other photometric surveys

This notebook will be pulling heavily from gaia/mordor_survey_code/mega_table_joining.ipynb

In [1]:
from __future__ import print_function
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coord
from astropy import units as u
from astropy import constants as const
from astropy.table import Table, Column, vstack, join
#import scipy.interpolate as scinterp
import time
import pyvo
from astroquery.vizier import Vizier

from astroquery.gaia import Gaia
from astroquery.xmatch import XMatch


sys.path.append('../')

Could not import regions, which is required for some of the functionalities of this module.


In [2]:
Gaia.login(credentials_file='../Gaia_credentials.txt')

INFO: Login to gaia TAP server [astroquery.gaia.core]
OK
INFO: Login to gaia data server [astroquery.gaia.core]
OK


input_file='full_MORDOR_survey_1681141339.csv'
input_directory='/Users/BenKaiser/Desktop/MORDOR_Survey_forPaper/Goodman_spectra/'

I'm pretty sure I don't want to download the Gentile Fusillo catalogue ahead of time.
https://cdsarc.cds.unistra.fr/viz-bin/cat/J/MNRAS/508/3877

In [5]:
GentileFusillo_catname="J/MNRAS/508/3877"
gfviz=Vizier(catalog=GentileFusillo_catname,columns=['GaiaDR2'])
Vizier(catalog=GentileFusillo_catname).get_catalog_metadata()


AttributeError: 'VizierClass' object has no attribute 'get_catalog_metadata'

print(vizier)

In an attempt to join the future I have decided to consult AI to write this code for me. A Brave New World. I tried Chat GPT, Grok, and CoPilot. It looked like Copilot had the best script among the options. The code below was written by copilot.

#!/usr/bin/env python3
"""
Read a fixed‐width header spec from a file and
use it to load data into an Astropy Table.
"""

import re
import argparse
from astropy.io import ascii

def parse_header_spec(filename):
    """
    Parse a header‐spec file with lines like:
       1- 23  A23  ---  WDJname  ...
    Returns lists: col_starts, col_ends, col_names, col_units
    """
    col_starts, col_ends, col_names, col_units = [], [], [], []

    with open(filename, 'r') as fh:
        for line in fh:
            line = line.rstrip()
            # skip empty lines or separator line
            if not line.strip() or line.startswith('-'):
                continue

            # regex to grab start-end, format, unit, name
            m = re.match(r"\s*(\d+)-\s*(\d+)\s+\S+\s+(\S+)\s+(\S+)", line)
            if not m:
                continue

            start, end, unit, name = m.groups()
            # convert to 0‐based start, inclusive end
            col_starts.append(int(start) - 1)
            col_ends.append(int(end))
            col_names.append(name)
            col_units.append(None if unit == '---' else unit)

    return col_starts, col_ends, col_names, col_units

def main(filename):

    # parse the external spec
    starts, ends, names, units = parse_header_spec(filename)

    # read the data
    table = ascii.read(
        filename,
        format="fixed_width",
        col_starts=starts,
        col_ends=ends,
        names=names,
    )

    # show results
    print(table)
    print("Columns:", table.colnames)
    return table



header_file='maincat_headers.dat'
colnames=main(header_file)

#!/usr/bin/env python3
"""
Use Gaia eDR3’s external catalog cross-matches to find
Pan-STARRS DR1 photometry for a list of Gaia source_ids.
"""

from astroquery.gaia import Gaia
from astropy.table import Table, vstack
import numpy as np
from astropy.io import ascii

# 1. Load your Gentile Fusillo WD table (eDR3) with a 'source_id' column.
#    Replace this with however you ingest your local catalogue.
#gf_cat = Table.read("maincat.dat",format='ascii.tab',colnames=colnames.colnames,fast_reader=False)  
gf_cat = Table.read("maincat.dat",format='ascii.fixed_width',data_start=0)  
#gf_cat = ascii.read("maincat.dat",format='fixed_width_no_header')  

print('fixed_width tried')

So apparently the catalogue is getting read in with a single column name, and actually maybe only a single column...

for index,thing in enumerate(gf_cat[0]):
    print('++++')
    print(index,thing,'\n')

Yep, there's only one column as read in presently. So not tabs I guess.

print(len(gf_cat.colnames))

So fixed_width also yields a single column table... groan

gf_cat.pprint(1)

print(gf_cat.colnames)

adql = f"""
    SELECT
      nb.                           AS gaia_source_id,
      ps.objID                               AS ps1_objid,
      ps.raMean                              AS ra_ps1,
      ps.decMean                             AS dec_ps1,
      ps.gMeanPSFMag                         AS g_psf,
      ps.rMeanPSFMag                         AS r_psf,
      ps.iMeanPSFMag                         AS i_psf,
      ps.zMeanPSFMag                         AS z_psf,
      ps.yMeanPSFMag                         AS y_psf
    FROM gaiadr3.panstarrs1_gaiadr3_best_neighbour AS nb
    JOIN gaiadr3.panstarrs1_gaiadr3_original_valid AS ps
      ON nb.original_ext_source_id = ps.objID
    WHERE nb.source_id IN ({id_list})
    """

rename_cols=zip(gf_cat.colnames,colnames)
for old,new in rename_cols:
    gf_cat.rename_column(old,new)


colnames_manual=['WDJname','GaiaEDR3','GaiaDR2',
                 'EDR3Name', 'RAdeg', 'e_RAdeg', 'DEdeg', 'e_DEdeg',
                 'Plx', 'e_Plx', 'RPlx',
                 'ZPcor', 'Pwd', 'density', 'SolID', 'RandomI', 'Epoch',  'PM', 'e_PM',
                 'pmRA', 'e_pmRA', 'pmDE', 'e_pmDE', 'RADEcor', 'RAPlxcor', 'RApmRAcor', 'RApmDEcor', 'DEPlxcor',
                 'DEpmRAcor', 'DEpmDEcor', 'PlxpmRAcor', 'PlxpmDEcor',
                 'pmRApmDEcor', 'NAL', 'NAC', 'NgAL', 'NbAL', 'gofAL', 'chi2AL', 'epsi',
                 'sepsi', 'Solved', 'nueff', 'pscol', 'e_pscol'
                 'RApscolCorr', 'DEpscolCorr', 'PlxpscolCorr', 'pmRApscolCorr', 'pmDEpscolCorr', 'MatchObsA', 'Nper',
                 'amax', 'MatchObs', 'NewMatchObs', 'MatchObsrm',
                 'IPDgofha', 'IPDgofhp', 'IPDfmp', 'IPDfow', 'RUWE', 'SDSk1', 'SDSk2', 'SDSk3',
                 'SDSk4', 'SDMk1', 'SDMk2', 'SDMk3', 'SDMk4', 'o_Gmag',  'FG', 'e_FG',
                 'RFG', 'Gmag', 'e_Gmag', 'FGCorr', 'GmagCorr', 'e_GmagCorr', 'o_BPmag', 'FBP',
                 'e_FBP', 'RFBP', 'BPmag', 'e_BPmag', 'o_RPmag', 'FRP', 'e_FRP', 'RFRP', 'RPmag', 'e_RPmag', 'NBPcont', 'NBPblend',
                 'NRPcont', 'NRPblend', 'E(BP/RP)','E(BP/RP)Corr', 'GMAG', 'BP-RP',
                 'BP-G', 'G-RP', 'GLON', 'GLAT', 'ELON', 'ELAT', 'ExFluxErr', 'meanAV', 'minAV', 'maxAV', 'TeffH', 'e_TeffH',
                 'loggH', 'e_loggH', 'MassH', 'e_MassH', 'chisqH', 'TeffHe', 'e_TeffHe', 'loggHe', 'e_loggHe', 'MassHe',
                 'e_MassHe', 'chisqHe', 'Teffmix', 'e_Teffmix', 'loggmix', 'e_loggmix', 'Massmix', 'e_Massmix',
                 'chisqmix', 'rgeo', 'b_rgeo', 'B_rgeo', 'rpgeo', 'b_rpgeo', 'B_rpgeo', 'fidel-v1',
                 'SDSS12', 'umag','e_umag', 'gmag', 'e_gmag', 'rmag', 'e_rmag', 'imag', 'e_imag', 'zmag', 'e_zmag', 'SDSSsep',
                 'SDSSspec']

for name in gf_cat.colnames:
    print(name)
print(len(gf_cat.colnames))
for name in colnames:
    print(name)
print(len(colnames))
print(len(gf_cat))

# *Trying this once again after realizing the download-reupload route was moronic*

In [ ]:

# 1. Always set row_limit globally (or per-instance) 
Vizier.ROW_LIMIT = -1    # no cap; remove if you only want top N rows
Vizier.TIMEOUT   = 60    # seconds

# 2. Build a Vizier instance with column selection + filters
v = Vizier(
    columns = [
        "WDJname", "Gmag", "e_Gmag",
        "Plx",    "e_Plx",
        "BP-RP",  "RUWE"
    ],
    column_filters = {
        "Gmag": "[10,15]",
        "Plx":  "[5,]",
        "RUWE": "[,1.4]"
    },
    row_limit = -1         # override any default row cap
)

# 3. Query the catalog by its Vizier code
result_tables = v.query_catalog("J/ApJ/921/91")
gf_filtered = result_tables[0]  # first (and only) table

print(f"Selected objects: {len(gf_filtered)}")
print(gf_filtered[:5])           # preview first 5 rows


In [ ]:
gf_cat['GaiaEDR3'][0:3]